In [ ]:
from google.colab import files
import os
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
uploaded_filenames = []
try:
  uploaded = files.upload()
  for name, data in uploaded.items():
    uploaded_filenames.append(name)
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=name, length=len(data)))
except Exception as e:
  print(f"Error during file upload: {e}")

if uploaded_filenames:
    print(f"Uploaded file(s): {', '.join(uploaded_filenames)}")
else:
    print("No files were uploaded.")


In [ ]:
log_file_path = ''

if 'uploaded_filenames' in locals() and uploaded_filenames:
    log_file_name = uploaded_filenames[0]
    log_file_path = os.path.join('/content/', log_file_name)
    print(f"Using uploaded file: {log_file_path}")
else:
    log_file_path = '/content/game_log.txt'
    print(f"No file uploaded, using default file: {log_file_path}")

if os.path.exists(log_file_path):
    with open(log_file_path, 'r') as file:
        game_log_content = file.readlines()

    print("First 10 lines of the game log:")
    for i, line in enumerate(game_log_content):
        if i >= 10:
            break
        print(line.strip())
else:
    print(f"Error: The log file '{log_file_path}' does not exist.")
    game_log_content = []


# Extract Winnings


In [ ]:
#Enter Your Bot Names here to analyse performance in ""
MY_BOT_NAMES = []

botA_winnings = []

for line in game_log_content:
    for name in MY_BOT_NAMES:
        if f'{name} awarded' in line:
            match = re.search(fr'{name} awarded (-?\d+)', line)
            if match:
                botA_winnings.append(int(match.group(1)))
                break

print("First 10 entries of Your winnings:", botA_winnings[:10])


# Calculate Winnings Statistics


In [ ]:
total_winnings_botA = sum(botA_winnings)
average_winnings_botA = total_winnings_botA / len(botA_winnings) if botA_winnings else 0
min_winnings_botA = min(botA_winnings) if botA_winnings else 0
max_winnings_botA = max(botA_winnings) if botA_winnings else 0
positive_wins_botA = sum(1 for w in botA_winnings if w > 0)


print("Your Winnings Statistics:")
print(f"  Total Winnings: {total_winnings_botA}")
print(f"  Average Winnings per Game: {average_winnings_botA:.2f}")
print(f"  Minimum Winnings: {min_winnings_botA}")
print(f"  Maximum Winnings: {max_winnings_botA}")
print(f"  Number of Positive Wins: {positive_wins_botA}")


# Plot Winnings Histograms


In [ ]:
min_w = min(botA_winnings)
max_w = max(botA_winnings)
bins_freq = range(int(min_w / 500) * 500, int(max_w / 500) * 500 + 501, 500)

winnings_distribution = defaultdict(int)

for winning in botA_winnings:
    found_bin = False
    for i in range(len(bins_freq) - 1):
        if bins_freq[i] <= winning < bins_freq[i+1]:
            winnings_distribution[f'{bins_freq[i]} to {bins_freq[i+1]-1}'] += 1
            found_bin = True
            break

    if not found_bin:
        if winning < bins_freq[0]:
            winnings_distribution[f'Less than {bins_freq[0]}'] += 1
        else:
            winnings_distribution[f'Greater than or equal to {bins_freq[-1]}'] += 1

sorted_intervals = sorted(winnings_distribution.keys(),
                          key=lambda x: int(x.split(' ')[0]) if x.split(' ')[0].replace('-', '').isdigit() else -float('inf'))

print("Your Winnings Frequency Table (Intervals of 500):")
print("--------------------------------------------------")
print(f"| Interval             | Count   |")
print("--------------------------------------------------")
for interval in sorted_intervals:
    print(f"| {interval:<20} | {winnings_distribution[interval]:<7} |")
print("--------------------------------------------------")


In [ ]:


fig, axes = plt.subplots(1,1, figsize=(12, 5))

min_val = min(botA_winnings)
max_val = max(botA_winnings)
bins = range(int(min_val/500)*500, int(max_val/500)*500 + 501, 500)

axes.hist(botA_winnings, bins=bins, color='red', edgecolor='black')
axes.set_title('Distribution of BotA Winnings (bins=500)')
axes.set_xlabel('Winnings (bins of 500)')
axes.set_ylabel('Frequency')
axes.set_xticks(bins)
axes.tick_params(axis='x', rotation=45) # Rotate x-axis labels for readability

fig.suptitle('Winnings Distribution for BotA and BotB', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
cumulative_winnings = [sum(botA_winnings[:i+1]) for i in range(len(botA_winnings))]

rounds = range(1, len(botA_winnings) + 1)

plt.figure(figsize=(12, 6))
plt.plot(rounds, cumulative_winnings, label='Net Winnings', color='green')

plt.title('Net Winnings Over Rounds')
plt.xlabel('Number of Rounds')
plt.ylabel('Net Winnings')
plt.grid(True)
plt.axhline(0, color='red', linestyle='--', linewidth=0.8) # Add a line for zero winnings
plt.legend()
plt.show()


In [ ]:
hand_outcomes = []
current_hand_data = {'folded': False, 'winnings': None}

for line in game_log_content:
    if line.startswith('Round #'):
        if current_hand_data['winnings'] is not None:
            if current_hand_data['winnings'] > 0:
                current_hand_data['folded'] = False
            hand_outcomes.append(current_hand_data.copy())
        current_hand_data = {'folded': False, 'winnings': None}

    for name in MY_BOT_NAMES:
        if f'{name} folds' in line:
            current_hand_data['folded'] = True
            break

    for name in MY_BOT_NAMES:
        match = re.search(fr'{name} awarded (-?\d+)', line)
        if match:
            current_hand_data['winnings'] = int(match.group(1))
            if current_hand_data['winnings'] > 0:
                current_hand_data['folded'] = False
            break

if current_hand_data['winnings'] is not None:
    if current_hand_data['winnings'] > 0:
        current_hand_data['folded'] = False
    hand_outcomes.append(current_hand_data.copy())

print("First 5 entries of hand_outcomes (corrected and re-evaluated for folds):")
for i, entry in enumerate(hand_outcomes):
    if i >= 5:
        break
    print(entry)


In [ ]:
folds_in_won_hands = 0
folds_in_lost_hands = 0

for hand in hand_outcomes:
    if hand['folded']:
        if hand['winnings'] > 0:
            folds_in_won_hands += 1
        elif hand['winnings'] < 0:
            folds_in_lost_hands += 1


print(f"Number of times Your Bot folded in a lost hand: {folds_in_lost_hands}")


In [ ]:
folded_lost_winnings = [hand['winnings'] for hand in hand_outcomes if hand['folded'] and hand['winnings'] < 0]

if folded_lost_winnings:
    bins = range(-5000, 51, 50)

    plt.figure(figsize=(12, 6))
    plt.hist(folded_lost_winnings, bins=bins, color='skyblue', edgecolor='black')
    plt.title('Distribution of Winnings when Folding in Lost Hands (-5000 to 0, bins=50)')
    plt.xlabel('Winnings (bins of 50)')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No instances of folding in a lost hand to plot the winnings distribution.")


In [ ]:
folded_lost_winnings = [hand['winnings'] for hand in hand_outcomes if hand['folded'] and hand['winnings'] < 0]

if folded_lost_winnings:
    min_f_w = min(folded_lost_winnings)
    max_f_w = max(folded_lost_winnings)

    bins_freq_folded_lost_50 = range(min(int(min_f_w / 50) * 50, -5000), max(int(max_f_w / 50) * 50, 0) + 51, 50)

    winnings_distribution_folded_lost_50 = defaultdict(int)

    for winning in folded_lost_winnings:
        found_bin = False
        for i in range(len(bins_freq_folded_lost_50) - 1):
            if bins_freq_folded_lost_50[i] <= winning < bins_freq_folded_lost_50[i+1]:
                winnings_distribution_folded_lost_50[f'{bins_freq_folded_lost_50[i]} to {bins_freq_folded_lost_50[i+1]-1}'] += 1
                found_bin = True
                break

        if not found_bin:
            if winning < bins_freq_folded_lost_50[0]:
                winnings_distribution_folded_lost_50[f'Less than {bins_freq_folded_lost_50[0]}'] += 1
            elif winning >= bins_freq_folded_lost_50[-1]:
                 winnings_distribution_folded_lost_50[f'Greater than or equal to {bins_freq_folded_lost_50[-1]}'] += 1


    sorted_intervals_folded_lost_50 = sorted(winnings_distribution_folded_lost_50.keys(),
                                  key=lambda x: int(x.split(' ')[0]) if x.split(' ')[0].replace('-', '').isdigit() else -float('inf'))

    print("Your Winnings Frequency Table for Folds in Lost Hands (Intervals of 50):")
    print("--------------------------------------------------")
    print(f"| Interval             | Count   |")
    print("--------------------------------------------------")
    for interval in sorted_intervals_folded_lost_50:
        print(f"| {interval:<20} | {winnings_distribution_folded_lost_50[interval]:<7} |")
    print("--------------------------------------------------")
else:
    print("No of instances of folding in a lost hand to analyze for winnings distribution.")


In [ ]:
hands = range(1, len(botA_winnings) + 1)

plt.figure(figsize=(15, 7))
plt.bar(hands, botA_winnings[:], color='skyblue')
plt.xlabel('Hand Number')
plt.ylabel('Winnings')
plt.title('Winnings per Hand for (Hands 600 onwards)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
folded_won_count = 0
folded_lost_count = 0
not_folded_won_count = 0
not_folded_lost_count = 0

folded_won_winnings = []
folded_lost_winnings = []
not_folded_won_winnings = []
not_folded_lost_winnings = []

for hand in hand_outcomes:
    if hand['folded']:
        if hand['winnings'] > 0:
            folded_won_count += 1
            folded_won_winnings.append(hand['winnings'])
        elif hand['winnings'] < 0:
            folded_lost_count += 1
            folded_lost_winnings.append(hand['winnings'])
    else:
        if hand['winnings'] > 0:
            not_folded_won_count += 1
            not_folded_won_winnings.append(hand['winnings'])
        elif hand['winnings'] < 0:
            not_folded_lost_count += 1
            not_folded_lost_winnings.append(hand['winnings'])

avg_folded_won = sum(folded_won_winnings) / len(folded_won_winnings) if folded_won_winnings else 0
avg_folded_lost = sum(folded_lost_winnings) / len(folded_lost_winnings) if folded_lost_winnings else 0
avg_not_folded_won = sum(not_folded_won_winnings) / len(not_folded_won_winnings) if not_folded_won_winnings else 0
avg_not_folded_lost = sum(not_folded_lost_winnings) / len(not_folded_lost_winnings) if not_folded_lost_winnings else 0

print("Temperament Table :")
print("---------------------------------------------------------------------------------")
print(f"|                 | Count (Won)     | Avg. Winnings (Won) || Count (Lost)    | Avg. Winnings (Lost) |")
print("---------------------------------------------------------------------------------")
print(f"| Folded          | {folded_won_count:<15} | {avg_folded_won:<19.2f} || {folded_lost_count:<15} | {avg_folded_lost:<20.2f} |")
print(f"| Not Folded      | {not_folded_won_count:<15} | {avg_not_folded_won:<19.2f} || {not_folded_lost_count:<15} | {avg_not_folded_lost:<20.2f} |")
print("---------------------------------------------------------------------------------")


# **Reasoning**:


In [ ]:
botA_hole_cards = []

for line in game_log_content:
    for name in MY_BOT_NAMES:
        if f'{name} received' in line:
            match = re.search(r'\[([2-9TJQKA][scdh])\s+([2-9TJQKA][scdh])\]', line)
            if match:
                botA_hole_cards.append([match.group(1), match.group(2)])
                break

if len(botA_hole_cards) != len(botA_winnings):
    print(f"Warning: Mismatch in number of extracted hole card hands ({len(botA_hole_cards)}) and winnings ({len(botA_winnings)}).")
else:
    print("Number of extracted hole card hands and winnings match.")

hole_card_winnings_pairs = list(zip(botA_hole_cards, botA_winnings))

print("\nFirst 5 entries of (Hole Cards, Winnings) pairs:")
for i, (cards, winnings) in enumerate(hole_card_winnings_pairs):
    if i >= 5:
        break
    print(f"  Cards: {cards}, Winnings: {winnings}")


# Process and Aggregate Card Ranks


In [ ]:
rank_map = {
    '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9,
    'T': 10, 'J': 11, 'Q': 12, 'K': 13, 'A': 14
}

winnings_by_card_pair = defaultdict(list)

for cards, winnings in hole_card_winnings_pairs:
    rank1 = cards[0][0] # Get the first character of the card string (e.g., 'K' from 'Ks')
    rank2 = cards[1][0] # Get the first character of the card string (e.g., '4' from '4s')

    if rank_map[rank1] < rank_map[rank2]:
        normalized_card_pair = (rank1, rank2)
    else:
        normalized_card_pair = (rank2, rank1)

    winnings_by_card_pair[normalized_card_pair].append(winnings)

average_winnings_by_card_pair = {}

for card_pair, winnings_list in winnings_by_card_pair.items():
    average_winnings_by_card_pair[card_pair] = sum(winnings_list) / len(winnings_list)

print(f"Number of unique card pairs found: {len(average_winnings_by_card_pair)}")
print("\nFirst 5 entries of Average Winnings by Card Pair:")
for i, (pair, avg_winnings) in enumerate(average_winnings_by_card_pair.items()):
    if i >= 5:
        break
    print(f"  {pair}: {avg_winnings:.2f}")


In [ ]:
ranks_ordered = ['2', '3', '4', '5', '6', '7', '8', '9', 'T', 'J', 'Q', 'K', 'A']


heatmap_data = pd.DataFrame(np.nan, index=ranks_ordered, columns=ranks_ordered)


for (rank1, rank2), avg_winnings in average_winnings_by_card_pair.items():
    heatmap_data.loc[rank1, rank2] = avg_winnings
    if rank1 != rank2:
        heatmap_data.loc[rank2, rank1] = avg_winnings

print("Heatmap Data (first 5x5 block):\n", heatmap_data.iloc[:5, :5])
print("\nHeatmap Data (last 5x5 block):\n", heatmap_data.iloc[-5:, -5:])


In [ ]:
plt.figure(figsize=(10, 8))

ax = sns.heatmap(heatmap_data,
            annot=True,
            fmt=".1f",
            cmap="RdYlGn",
            linewidths=.5,
            linecolor='black',
            cbar_kws={'label': 'Average Winnings'},
            vmin=-1000,
            vmax=1000)

plt.title('Average Winnings by Hole Card Pair', fontsize=16)
plt.xlabel('Card Rank', fontsize=12)
plt.ylabel('Card Rank', fontsize=12)

ax.invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
opponent_name = None

for line in game_log_content:
    if 'vs' in line:
        parts = line.split('vs')
        if len(parts) == 2:
            player1 = parts[0].strip().split(' ')[-1] # Get last word before 'vs'
            player2 = parts[1].strip().split(' ')[0] # Get first word after 'vs'

            if player1 not in MY_BOT_NAMES:
                opponent_name = player1
            elif player2 not in MY_BOT_NAMES:
                opponent_name = player2

            if opponent_name:
                print(f"Identified Opponent: {opponent_name}")
                break

if opponent_name is None:
    print("Could not identify opponent's name from the log.")


In [ ]:
hand_outcomes = []
current_hand_data = {'folded': False, 'winnings': None, 'opponent_folded': False}

for line in game_log_content:
    if line.startswith('Round #'):
        if current_hand_data['winnings'] is not None:
            if current_hand_data['winnings'] > 0:
                current_hand_data['folded'] = False
            hand_outcomes.append(current_hand_data.copy())
        current_hand_data = {'folded': False, 'winnings': None, 'opponent_folded': False}

    for name in MY_BOT_NAMES:
        if f'{name} folds' in line:
            current_hand_data['folded'] = True
            break

    if opponent_name and f'{opponent_name} folds' in line:
        current_hand_data['opponent_folded'] = True

    for name in MY_BOT_NAMES:
        match = re.search(fr'{name} awarded (-?\d+)', line)
        if match:
            current_hand_data['winnings'] = int(match.group(1))
            if current_hand_data['winnings'] > 0:
                current_hand_data['folded'] = False
            break

if current_hand_data['winnings'] is not None:
    if current_hand_data['winnings'] > 0:
        current_hand_data['folded'] = False
    hand_outcomes.append(current_hand_data.copy())

print("First 5 entries of hand_outcomes (updated with opponent_folded):")
for i, entry in enumerate(hand_outcomes):
    if i >= 5:
        break
    print(entry)


In [ ]:
folded_opponent_folded_count = 0
folded_opponent_not_folded_count = 0
not_folded_opponent_folded_count = 0
not_folded_opponent_not_folded_count = 0

folded_opponent_folded_winnings = []
folded_opponent_not_folded_winnings = []
not_folded_opponent_folded_winnings = []
not_folded_opponent_not_folded_winnings = []

for hand in hand_outcomes:
    if hand['folded']:
        if hand['opponent_folded']:
            folded_opponent_folded_count += 1
            folded_opponent_folded_winnings.append(hand['winnings'])
        else:
            folded_opponent_not_folded_count += 1
            folded_opponent_not_folded_winnings.append(hand['winnings'])
    else:
        if hand['opponent_folded']:
            not_folded_opponent_folded_count += 1
            not_folded_opponent_folded_winnings.append(hand['winnings'])
        else:
            not_folded_opponent_not_folded_count += 1
            not_folded_opponent_not_folded_winnings.append(hand['winnings'])

avg_folded_opponent_folded = sum(folded_opponent_folded_winnings) / len(folded_opponent_folded_winnings) if folded_opponent_folded_winnings else 0
avg_folded_opponent_not_folded = sum(folded_opponent_not_folded_winnings) / len(folded_opponent_not_folded_winnings) if folded_opponent_not_folded_winnings else 0
avg_not_folded_opponent_folded = sum(not_folded_opponent_folded_winnings) / len(not_folded_opponent_folded_winnings) if not_folded_opponent_folded_winnings else 0
avg_not_folded_opponent_not_folded = sum(not_folded_opponent_not_folded_winnings) / len(not_folded_opponent_not_folded_winnings) if not_folded_opponent_not_folded_winnings else 0

print("Detailed Temperament Table :")
print("-------------------------------------------------------------------------------------------------------------")
print(f"| You Folded | Opponent Folded | Count           | Average Winnings  |")
print("-------------------------------------------------------------------------------------------------------------")
print(f"| Yes         | No              | {folded_opponent_not_folded_count:<15} | {avg_folded_opponent_not_folded:<23.2f} |")
print(f"| No          | Yes             | {not_folded_opponent_folded_count:<15} | {avg_not_folded_opponent_folded:<23.2f} |")
print(f"| Showdown    | Showdown        | {not_folded_opponent_not_folded_count:<15} | {avg_not_folded_opponent_not_folded:<23.2f} |")
print("-------------------------------------------------------------------------------------------------------------")

